In [0]:
from pyspark.sql import SparkSession, functions as F
import json, uuid, requests, os
from datetime import datetime
from databricks.sdk import WorkspaceClient

spark = SparkSession.builder.getOrCreate()
w = WorkspaceClient()

# 1. Aggregate symptom frequencies across all records
symptom_df = spark.sql("""
    SELECT explode(symptoms) as symptom, 
           count(*) as frequency,
           collect_set(hospital_id) as hospitals
    FROM main.vaidya.patient_records
    WHERE timestamp >= current_timestamp() - interval 7 days
    GROUP BY symptom
    ORDER BY frequency DESC
    LIMIT 20
""")
symptom_df.show()

# 2. Use Llama to generate human-readable insight
top_symptoms = [(r["symptom"], r["frequency"]) for r in symptom_df.collect()]
prompt = f"""You are a public health AI for India.
Given the top symptoms seen across hospitals this week:
{json.dumps(top_symptoms)}

Generate 3 concise alert bullets for doctors. Each should be:
- Actionable (what to watch for)
- Evidence-based (based on the symptom data above)
- India-specific (mention seasonal context if relevant for March)

Respond as JSON: {{"alerts": ["...", "...", "..."]}}"""

# Reuse the same AI Gateway pattern
from databricks.sdk.service.serving import ChatMessage
headers = w.config.authenticate()
resp = requests.post(
    f"{os.environ.get('LLM_OPENAI_BASE_URL')}/chat/completions",
    headers={**headers, "Content-Type": "application/json"},
    json={
        "model": "databricks-llama-4-maverick",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 512, "temperature": 0.3
    }
)
content = resp.json()["choices"][0]["message"]["content"]
content = content.strip().lstrip("```json").rstrip("```").strip()
alerts_data = json.loads(content)

# 3. Write to health_alerts Delta table
from pyspark.sql import Row
rows = [Row(
    alert_id=str(uuid.uuid4()),
    generated_at=datetime.now(),
    alert_type="symptom_trend",
    insight_text=alert,
    top_symptoms=[s[0] for s in top_symptoms[:5]],
    affected_hospitals=list({h for r in symptom_df.collect() for h in r["hospitals"]}),
    severity="INFO"
) for alert in alerts_data["alerts"]]

alert_df = spark.createDataFrame(rows)
alert_df.write.mode("append").saveAsTable("main.vaidya.health_alerts")
print("Alerts written:", len(rows))